# 03 — Benchmark CatBoost

Este notebook construye un **benchmark de mayor complejidad** para comparar contra el modelo principal:

> **WOE + Regresión Logística de 18 variables**

El objetivo **no es reemplazar automáticamente** la regresión logística si CatBoost obtiene un Gini ligeramente superior.

La pregunta que queremos responder es:

> **¿Cuánto poder predictivo adicional obtenemos al utilizar un modelo más complejo y más variables?**

La comparación se hará bajo las mismas reglas:

- misma base entregada para el reto;
- mismo TARGET;
- mismas exclusiones de leakage;
- mismo split temporal;
- TRAIN aprende;
- VALIDATION decide;
- OOT confirma.

## Qué NO haremos

Este benchmark no incluirá:

- Optuna;
- Grid Search masivo;
- Feature Engineering adicional;
- selección retrospectiva usando OOT.

Queremos un CatBoost **razonable y reproducible**, no una competencia de hiperparámetros.

## Criterio de decisión

El resultado se evaluará junto con:

- Gini / AUC / KS;
- estabilidad temporal;
- número de variables;
- explicabilidad;
- auditabilidad;
- complejidad de implementación.

La regresión logística seguirá siendo el modelo principal salvo que el benchmark muestre una mejora predictiva suficientemente grande como para justificar el aumento de complejidad.


## 1. Librerías y configuración

La ruta asume la misma estructura del notebook logístico:

`data/raw/base_analytics_lab_061628.parquet`


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    brier_score_loss
)

from catboost import CatBoostClassifier


# ============================================================
# RUTAS
# ============================================================

ROOT = Path.cwd().resolve()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent

DATA_PATH = (
    ROOT
    / "data"
    / "raw"
    / "base_analytics_lab_061628.parquet"
)


# ============================================================
# CONFIGURACIÓN
# ============================================================

TARGET = "TARGET"
TIEMPO = "CODMES"

RANDOM_SEED = 42

print("Ruta:", DATA_PATH)


Ruta: C:\Users\Manolo\Desktop\Compartamos definitivo\Entrega_Compartamos_Final\data\raw\base_analytics_lab_061628.parquet


## 2. Carga de datos

La versión del reto contiene 49 columnas.

CatBoost trabajará directamente con las variables originales permitidas, sin WOE.

Esto hace que la comparación también muestre el costo de complejidad del modelo:

- Logística final: **18 variables WOE**
- CatBoost benchmark: conjunto amplio de predictores disponibles


In [2]:
df = pd.read_parquet(
    DATA_PATH
).copy()

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

display(
    df.head()
)


Filas: 60000
Columnas: 49


,CODMES,TARGET,MEDIDA1_MORA_INTERNA_U6M,MEDIDA1_MORA_INTERNA_U12M,MEDIDA1_MORA_INTERNA_U1M,MEDIDA2_MORA_INTERNA_U12M,MEDIDA2_MORA_INTERNA_U6M,MEDIDA3_MORA_INTERNA_U12M,MEDIDA1_APALANCAMIENTO_INTERNO_U6M,MEDIDA2_APALANCAMIENTO_INTERNO_U6M,...,MEDIDA2_PRODUCTOS_SSFF_U3M,MEDIDA1_LINEA_TARJETACREDITO,PREDICCION_MODELO_ANTERIOR,PROXY_TARGET_6M,MEDIDA1_TRX_CASHIN_U3M,MEDIDA2_TRX_TRANSFER_U1M,MEDIDA3_TRX_QR_U1M,MEDIDA4_TRX_DEBITO_U1M,MEDIDA5_TRX_ECOM_U1M,MEDIDA6_TRX_APP_U1M
0,202308,0,0,0,-1,99,99,0,0.853261,1.000000,...,2.0,NaN,0.250741,0,26.740000,NaN,NaN,1.0,51.860001,1
1,202401,1,4,4,1,99,0,5,0.706940,0.897245,...,5.0,NaN,0.605026,0,71.269997,89.379997,NaN,6.0,49.099998,1
2,202402,0,3,3,3,99,0,6,0.392161,0.869825,...,2.0,NaN,0.601767,0,40.180000,136.169998,NaN,1.0,153.330002,1
3,202307,1,21,21,21,0,0,2,0.706261,1.049905,...,3.0,NaN,0.665567,0,739.659973,NaN,NaN,2.0,176.559998,0
4,202307,1,0,0,0,99,99,0,1.000000,1.238984,...,4.0,NaN,0.564676,1,374.079987,291.910004,NaN,3.0,59.529999,0


## 3. Exclusiones

Para que la comparación sea justa, se aplican las mismas exclusiones estructurales del desarrollo logístico.

No utilizamos:

- `TARGET`: es la variable objetivo.
- `CODMES`: solo define el tiempo.
- `PREDICCION_MODELO_ANTERIOR`: benchmark anterior, no predictor del nuevo modelo.
- `PROXY_TARGET_6M`: contiene información futura y produciría leakage.

Por tanto, CatBoost trabaja con los predictores restantes disponibles en la base.


In [3]:
# ============================================================
# VARIABLES EXCLUIDAS
# ============================================================

variables_excluir = [
    TARGET,
    TIEMPO,
    "PREDICCION_MODELO_ANTERIOR",
    "PROXY_TARGET_6M"
]


variables_catboost = [
    columna
    for columna in df.columns
    if columna not in variables_excluir
]


print(
    "Predictores CatBoost:",
    len(variables_catboost)
)

print(
    "\nPrimeras variables:"
)

print(
    variables_catboost[:10]
)


Predictores CatBoost: 45

Primeras variables:
['MEDIDA1_MORA_INTERNA_U6M', 'MEDIDA1_MORA_INTERNA_U12M', 'MEDIDA1_MORA_INTERNA_U1M', 'MEDIDA2_MORA_INTERNA_U12M', 'MEDIDA2_MORA_INTERNA_U6M', 'MEDIDA3_MORA_INTERNA_U12M', 'MEDIDA1_APALANCAMIENTO_INTERNO_U6M', 'MEDIDA2_APALANCAMIENTO_INTERNO_U6M', 'SOCIODEMOGRAFICO_1', 'MEDIDA2_APALANCAMIENTO_INTERNO_U12M']


## 4. Split temporal

Usamos exactamente el mismo split del modelo logístico:

- TRAIN: hasta `202401`
- VALIDATION: `202402–202403`
- OOT: `202404–202405`

OOT no se utilizará para seleccionar hiperparámetros ni para detener el entrenamiento.


In [4]:
train = df[
    df[TIEMPO] <= 202401
].copy()

validation = df[
    df[TIEMPO].isin(
        [202402, 202403]
    )
].copy()

oot = df[
    df[TIEMPO].isin(
        [202404, 202405]
    )
].copy()


resumen_split = pd.DataFrame({

    "Muestra": [
        "TRAIN",
        "VALIDATION",
        "OOT"
    ],

    "Filas": [
        len(train),
        len(validation),
        len(oot)
    ],

    "Bad_Rate_%": [
        train[TARGET].mean() * 100,
        validation[TARGET].mean() * 100,
        oot[TARGET].mean() * 100
    ]
})


display(
    resumen_split.round(2)
)


,Muestra,Filas,Bad_Rate_%
0,TRAIN,38462,16.02
1,VALIDATION,10583,14.95
2,OOT,10955,14.13


## 5. Preparación para CatBoost

CatBoost puede trabajar directamente con valores faltantes en variables numéricas.

Además, algunas variables de la base son numéricas por codificación, pero representan pocos estados.

Para el benchmark aplicaremos una regla sencilla y reproducible:

> **Si una variable presenta 20 categorías o menos en TRAIN, se tratará como categórica.**

La decisión se aprende únicamente con TRAIN.

Para las variables categóricas:

- los valores se convierten a texto;
- los missing se representan como `__MISSING__`.

Los valores centinela como `99`, `60` o `333333344` **no se eliminan**. CatBoost puede aprender splits que los separen si aportan señal.


In [5]:
# ============================================================
# IDENTIFICACIÓN DE VARIABLES CATEGÓRICAS
# APRENDIDA SOLO CON TRAIN
# ============================================================

variables_categoricas = [
    variable
    for variable in variables_catboost
    if train[variable].nunique(
        dropna=False
    ) <= 20
]


variables_numericas = [
    variable
    for variable in variables_catboost
    if variable not in variables_categoricas
]


print(
    "Variables categóricas:",
    len(variables_categoricas)
)

print(
    "Variables numéricas:",
    len(variables_numericas)
)

print(
    "\nCategóricas detectadas:"
)

print(
    variables_categoricas
)


Variables categóricas: 19
Variables numéricas: 26

Categóricas detectadas:
['MEDIDA2_MORA_INTERNA_U12M', 'MEDIDA2_MORA_INTERNA_U6M', 'MEDIDA3_MORA_INTERNA_U12M', 'SOCIODEMOGRAFICO_1', 'SOCIODEMOGRAFICO_4', 'SOCIODEMOGRAFICO_2', 'MEDIDA1_MORA_SSFF_U12M', 'MEDIDA1_DESAPALANCAMIENTO_SSFF_U12M', 'MEDIDA3_APALANCAMIENTO_SSFF_U12M', 'MEDIDA3_APALANCAMIENTO_SSFF_U6M', 'MEDIDA1_PRODUCTOS_SSFF_U1M', 'MEDIDA1_ENTIDADES_SSFF_U12M', 'MEDIDA1_CALIFICACION_SSFF_U60M', 'MEDIDA1_DESAPALANCAMIENTO_INTERNO_U6M', 'MEDIDA1_DESAPALANCAMIENTO_INTERNO_U12M', 'MEDIDA2_PRODUCTOS_SSFF_U3M', 'MEDIDA3_TRX_QR_U1M', 'MEDIDA4_TRX_DEBITO_U1M', 'MEDIDA6_TRX_APP_U1M']


In [6]:
def preparar_catboost(base):

    X = base[
        variables_catboost
    ].copy()

    # Variables categóricas:
    # CatBoost requiere valores tipo string
    for variable in variables_categoricas:

        X[variable] = (
            X[variable]
            .astype("object")
            .where(
                X[variable].notna(),
                "__MISSING__"
            )
            .astype(str)
        )

    # Variables numéricas:
    # se conservan como numéricas y CatBoost
    # gestiona NaN de forma nativa
    for variable in variables_numericas:

        X[variable] = pd.to_numeric(
            X[variable],
            errors="coerce"
        )

    return X


X_train_cb = preparar_catboost(
    train
)

X_valid_cb = preparar_catboost(
    validation
)

X_oot_cb = preparar_catboost(
    oot
)


y_train_cb = (
    train[TARGET]
    .astype(int)
)

y_valid_cb = (
    validation[TARGET]
    .astype(int)
)

y_oot_cb = (
    oot[TARGET]
    .astype(int)
)


print("TRAIN:", X_train_cb.shape)
print("VALIDATION:", X_valid_cb.shape)
print("OOT:", X_oot_cb.shape)


TRAIN: (38462, 45)
VALIDATION: (10583, 45)
OOT: (10955, 45)


## 6. Modelo CatBoost benchmark

Usamos una configuración moderada.

No buscamos exprimir el último decimal de Gini.

Decisiones:

- `loss_function="Logloss"`: clasificación binaria probabilística.
- `eval_metric="AUC"`: el benchmark se detiene según capacidad de discriminación en VALIDATION.
- `iterations=1000`: máximo de árboles.
- `learning_rate=0.03`: aprendizaje gradual.
- `depth=6`: complejidad moderada.
- `l2_leaf_reg=5`: regularización.
- `early_stopping_rounds=80`: evita continuar cuando VALIDATION deja de mejorar.
- `use_best_model=True`: conserva la mejor iteración según VALIDATION.
- `random_seed=42`: reproducibilidad.

### Importante

VALIDATION participa en la selección de la mejor iteración.

OOT no participa en ninguna decisión.


In [7]:
# ============================================================
# CATBOOST BENCHMARK
# ============================================================

modelo_catboost = CatBoostClassifier(

    iterations=1000,

    learning_rate=0.03,

    depth=6,

    loss_function="Logloss",

    eval_metric="AUC",

    l2_leaf_reg=5,

    random_seed=RANDOM_SEED,

    allow_writing_files=False,

    verbose=100
)


modelo_catboost.fit(

    X_train_cb,

    y_train_cb,

    cat_features=variables_categoricas,

    eval_set=(
        X_valid_cb,
        y_valid_cb
    ),

    early_stopping_rounds=80,

    use_best_model=True
)


print(
    "Mejor iteración:",
    modelo_catboost.get_best_iteration()
)


0:	test: 0.7099247	best: 0.7099247 (0)	total: 245ms	remaining: 4m 4s
100:	test: 0.7964016	best: 0.7964016 (100)	total: 12.5s	remaining: 1m 50s
200:	test: 0.8019708	best: 0.8019708 (200)	total: 23.9s	remaining: 1m 34s
300:	test: 0.8048399	best: 0.8048399 (300)	total: 35.3s	remaining: 1m 22s
400:	test: 0.8056144	best: 0.8056551 (397)	total: 45.5s	remaining: 1m 7s
500:	test: 0.8065466	best: 0.8065466 (500)	total: 58.7s	remaining: 58.5s
600:	test: 0.8070965	best: 0.8071177 (598)	total: 1m 10s	remaining: 46.9s
700:	test: 0.8072976	best: 0.8073382 (694)	total: 1m 23s	remaining: 35.5s
800:	test: 0.8078527	best: 0.8078599 (798)	total: 1m 36s	remaining: 23.9s
900:	test: 0.8079875	best: 0.8080925 (861)	total: 1m 49s	remaining: 12s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.8080925409
bestIteration = 861

Shrink model to first 862 iterations.
Mejor iteración: 861


## 7. Performance de CatBoost

Calculamos las mismas métricas utilizadas en la regresión logística:

- AUC
- Gini
- KS
- Brier
- Bad Rate
- PD promedio

Así la comparación es directa.


In [8]:
def calcular_ks(
    y_real,
    probabilidades
):

    fpr, tpr, _ = roc_curve(
        y_real,
        probabilidades
    )

    return np.max(
        tpr - fpr
    )


def resumen_metricas(
    nombre,
    y_real,
    probabilidades
):

    auc = roc_auc_score(
        y_real,
        probabilidades
    )

    return {

        "Muestra": nombre,

        "AUC": auc,

        "Gini": (
            2 * auc - 1
        ),

        "KS": calcular_ks(
            y_real,
            probabilidades
        ),

        "Bad_Rate_%": (
            np.mean(y_real)
            * 100
        ),

        "PD_Promedio_%": (
            np.mean(probabilidades)
            * 100
        ),

        "Brier": brier_score_loss(
            y_real,
            probabilidades
        )
    }


prob_train_cb = (
    modelo_catboost
    .predict_proba(
        X_train_cb
    )[:, 1]
)

prob_valid_cb = (
    modelo_catboost
    .predict_proba(
        X_valid_cb
    )[:, 1]
)

prob_oot_cb = (
    modelo_catboost
    .predict_proba(
        X_oot_cb
    )[:, 1]
)


tabla_metricas_catboost = pd.DataFrame([

    resumen_metricas(
        "TRAIN",
        y_train_cb,
        prob_train_cb
    ),

    resumen_metricas(
        "VALIDATION",
        y_valid_cb,
        prob_valid_cb
    ),

    resumen_metricas(
        "OOT",
        y_oot_cb,
        prob_oot_cb
    )
])


display(
    tabla_metricas_catboost.round(6)
)


,Muestra,AUC,Gini,KS,Bad_Rate_%,PD_Promedio_%,Brier
0,TRAIN,0.863099,0.726198,0.554235,16.018408,16.139073,0.090702
1,VALIDATION,0.808093,0.616185,0.460563,14.948502,15.587270,0.100780
2,OOT,0.804570,0.609139,0.462571,14.130534,16.042732,0.096791


## 8. Comparación directa contra la Regresión Logística

Los resultados oficiales del modelo logístico final fueron:

| Muestra | Gini Logística | AUC Logística | KS Logística |
|---|---:|---:|---:|
| TRAIN | 0.599015 | 0.799507 | 0.445362 |
| VALIDATION | 0.581576 | 0.790788 | 0.432179 |
| OOT | 0.592741 | 0.796370 | 0.449504 |

Ahora calculamos cuánto Gini adicional aporta CatBoost.


In [9]:
metricas_logistica = pd.DataFrame({

    "Muestra": [
        "TRAIN",
        "VALIDATION",
        "OOT"
    ],

    "Gini_Logistica": [
        0.599015,
        0.581576,
        0.592741
    ],

    "AUC_Logistica": [
        0.799507,
        0.790788,
        0.796370
    ],

    "KS_Logistica": [
        0.445362,
        0.432179,
        0.449504
    ]
})


tabla_comparacion_modelos = (
    tabla_metricas_catboost[
        [
            "Muestra",
            "AUC",
            "Gini",
            "KS"
        ]
    ]
    .rename(
        columns={
            "AUC": "AUC_CatBoost",
            "Gini": "Gini_CatBoost",
            "KS": "KS_CatBoost"
        }
    )
    .merge(
        metricas_logistica,
        on="Muestra",
        how="left"
    )
)


tabla_comparacion_modelos[
    "Delta_Gini"
] = (
    tabla_comparacion_modelos[
        "Gini_CatBoost"
    ]
    -
    tabla_comparacion_modelos[
        "Gini_Logistica"
    ]
)


tabla_comparacion_modelos[
    "Delta_Gini_puntos"
] = (
    tabla_comparacion_modelos[
        "Delta_Gini"
    ]
    * 100
)


display(
    tabla_comparacion_modelos.round(6)
)


,Muestra,AUC_CatBoost,Gini_CatBoost,KS_CatBoost,Gini_Logistica,AUC_Logistica,KS_Logistica,Delta_Gini,Delta_Gini_puntos
0,TRAIN,0.863099,0.726198,0.554235,0.599015,0.799507,0.445362,0.127183,12.718348
1,VALIDATION,0.808093,0.616185,0.460563,0.581576,0.790788,0.432179,0.034609,3.460908
2,OOT,0.804570,0.609139,0.462571,0.592741,0.796370,0.449504,0.016398,1.639841


## 9. Importancia de variables

CatBoost permite calcular importancia global de variables.

Esta información sirve como referencia para saber qué señales utiliza el modelo complejo.

Sin embargo, **importancia no equivale a dirección del riesgo**.

Una importancia alta no nos dice directamente:

- qué valores aumentan el riesgo;
- qué valores reducen el riesgo;
- cómo cambia la tasa de malos entre grupos.

Ese es uno de los motivos por los que los bins WOE son más fáciles de comunicar en el modelo principal.


In [10]:
importancia_catboost = pd.DataFrame({

    "Variable": variables_catboost,

    "Importancia": (
        modelo_catboost
        .get_feature_importance()
    )
})


importancia_catboost = (
    importancia_catboost
    .sort_values(
        "Importancia",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    importancia_catboost.head(20).round(4)
)


,Variable,Importancia
0,MEDIDA1_MORA_INTERNA_U6M,7.4097
1,MEDIDA2_CALIFICACION_SSFF_U60M,5.8832
2,MEDIDA3_APALANCAMIENTO_INTERNO_U1M,5.7572
3,MEDIDA2_TRX_TRANSFER_U1M,4.5825
4,MEDIDA1_MORA_INTERNA_U1M,4.2278
5,MEDIDA2_APALANCAMIENTO_SSFF_U60M,4.1852
6,SOCIODEMOGRAFICO_1,3.8259
7,DEUDA_PROMEDIO_SSFF,3.2810
8,MEDIDA4_DEUDA_SSFF_U1M,3.1675
9,MEDIDA1_LINEA_TARJETACREDITO,3.1375


## 10. Sobre SHAP

SHAP puede ser útil para explicar un modelo complejo como CatBoost.

Sin embargo, para este benchmark no es necesario desarrollar un análisis SHAP completo.

La razón es que el objetivo del notebook es responder:

> **¿Cuánto desempeño adicional aporta CatBoost?**

Si CatBoost aporta una mejora pequeña, la comparación ya permite justificar la elección de una solución WOE + Logística más simple.

Si la mejora fuera muy grande, entonces sí tendría sentido profundizar posteriormente en SHAP y gobernanza del modelo complejo.


## 11. Criterio de interpretación

No utilizaremos una regla automática como:

> “gana el modelo con mayor Gini”.

La decisión debe considerar el tamaño de la mejora.

Ejemplo conceptual:

```text
Logística OOT Gini = 0.593
CatBoost  OOT Gini = 0.610

Mejora = 0.017 = 1.7 puntos de Gini
```

La pregunta sería:

> ¿1.7 puntos adicionales justifican pasar de 18 variables interpretables a un modelo más complejo con un conjunto mucho mayor de predictores?

La respuesta se justificará considerando conjuntamente:

- discriminación;
- estabilidad;
- explicabilidad;
- cantidad de variables;
- mantenibilidad;
- facilidad de implementación;
- gobierno del modelo.


## 12. Salidas que necesito revisar

Después de ejecutar este notebook, comparte:

1. `Mejor iteración`
2. `tabla_metricas_catboost`
3. `tabla_comparacion_modelos`
4. las primeras 20 filas de `importancia_catboost`

Con esas salidas podremos escribir la conclusión definitiva del benchmark y decidir exactamente cómo presentarlo.

### Recordatorio

El benchmark no modifica retrospectivamente:

- los bins WOE;
- las 18 variables de la logística;
- el cutoff `PD < 20%`.

Es una comparación final de alternativas.


# 13. Conclusión del benchmark

CatBoost obtuvo una discriminación superior a la Regresión Logística WOE:

| Muestra | Gini Logística | Gini CatBoost | Mejora CatBoost |
|---|---:|---:|---:|
| TRAIN | 0.5990 | 0.7262 | +12.72 pts |
| VALIDATION | 0.5816 | 0.6162 | +3.46 pts |
| OOT | 0.5927 | 0.6091 | **+1.64 pts** |

CatBoost mejora el desempeño fuera de tiempo, pero la ventaja en OOT es de aproximadamente **1.64 puntos de Gini**.

La Regresión Logística presenta además una brecha mucho menor entre TRAIN y VALIDATION:

- Logística: aproximadamente **1.74 puntos de Gini**
- CatBoost: aproximadamente **11.00 puntos de Gini**

CatBoost se mantiene bien entre VALIDATION y OOT, por lo que no se concluye que sea un modelo inestable. Sin embargo, la mayor brecha TRAIN–VALIDATION es consistente con su mayor complejidad.

## Coincidencia en las señales de riesgo

Las variables más importantes de CatBoost incluyen varias de las variables seleccionadas por la logística:

- `MEDIDA1_MORA_INTERNA_U6M`
- `MEDIDA2_CALIFICACION_SSFF_U60M`
- `MEDIDA3_APALANCAMIENTO_INTERNO_U1M`
- `MEDIDA1_MORA_INTERNA_U1M`
- `SOCIODEMOGRAFICO_1`
- `DEUDA_PROMEDIO_SSFF`
- `MEDIDA1_LINEA_TARJETACREDITO`
- `MEDIDA1_ANTIGUEDAD_SSFF`
- `MEDIDA1_MORA_SSFF_U36M`

Esto muestra que ambos enfoques identifican varias de las mismas fuentes principales de riesgo.

CatBoost también utiliza variables que la logística descartó. Esto no es contradictorio: CatBoost puede aprovechar interacciones y relaciones no lineales que una regresión logística WOE no necesariamente necesita o retiene como aporte incremental.

## Modelo seleccionado

Se mantiene **WOE + Regresión Logística de 18 variables** como modelo principal.

La decisión se basa en que:

- conserva una discriminación competitiva;
- muestra desempeño consistente en las muestras temporales;
- utiliza solo 18 variables;
- permite explicar la dirección del riesgo mediante bins, Bad Rates y WOE;
- facilita auditoría, implementación, monitoreo y mantenimiento.

Por tanto, se acepta una reducción aproximada de **1.64 puntos de Gini OOT** frente a CatBoost a cambio de una solución sustancialmente más parsimoniosa e interpretable.

> **Conclusión ejecutiva:** CatBoost mejora el Gini OOT en aproximadamente 1.6 puntos, pero la Regresión Logística WOE conserva un desempeño competitivo con mayor explicabilidad, menor complejidad y mayor facilidad de gobernanza.
